# YDL 2026 Day 4 — Titanic with **Linear Regression only**

**Task:** predict Titanic survival on the Kaggle test set using *only* `sklearn.linear_model.LinearRegression`.

Survival is binary, but the constraint forces a regressor. Pipeline:
1. Engineer features (Title, FamilySize, HasCabin, **Family/Group survival**, …).
2. Fill missing values smartly (grouped medians, not one global mean).
3. One-hot encode categoricals and scale.
4. Fit `LinearRegression`, then **tune a decision threshold** with fold-safe repeated CV.
5. Retrain on the full training set and write the submission.

**Key upgrade over the 0.770 baseline:** a *Family/Group survival* feature — for each passenger we look at the known outcomes of their relatives / ticket-mates (from the **train** labels only, never the passenger's own label, never test). This is honest feature engineering on the provided data, and it lifts honest CV from ~0.83 to ~0.85.

Competition: https://www.kaggle.com/competitions/titanic

## 1. Imports

In [1]:
# Only LinearRegression is allowed as the ML model; the rest is preprocessing/CV.
import os
import warnings
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression          # the ONLY ML model allowed
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RepeatedStratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
pd.set_option('display.max_columns', None)

## 2. Load data

In [2]:
# Notebook runs from notebooks/ (data in ../data) or from project root (data/).
DATA = '../data' if os.path.isdir('../data') else 'data'
SUB = '../submissions' if os.path.isdir('../submissions') else 'submissions'
os.makedirs(SUB, exist_ok=True)

train = pd.read_csv(os.path.join(DATA, 'train.csv'))
test = pd.read_csv(os.path.join(DATA, 'test.csv'))
passenger_id = test['PassengerId']            # keep for the final submission

print('train', train.shape, '| test', test.shape)
train.head()

train (891, 12) | test (418, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 3. EDA

In [3]:
# Where are the missing values, and what is the base survival rate?
print('Missing (train):'); print(train.isna().sum()[train.isna().sum() > 0])
print('\nMissing (test):'); print(test.isna().sum()[test.isna().sum() > 0])
print('\nOverall survival rate:', round(train['Survived'].mean(), 4))

Missing (train):
Age         177
Cabin       687
Embarked      2
dtype: int64

Missing (test):
Age       86
Fare       1
Cabin    327
dtype: int64

Overall survival rate: 0.3838


In [4]:
# Survival clearly depends on Sex, Pclass and Embarked -> good predictors.
print('By Sex:\n',      train.groupby('Sex')['Survived'].mean(), '\n')
print('By Pclass:\n',   train.groupby('Pclass')['Survived'].mean(), '\n')
print('By Embarked:\n', train.groupby('Embarked')['Survived'].mean())

By Sex:
 Sex
female    0.742038
male      0.188908
Name: Survived, dtype: float64 

By Pclass:
 Pclass
1    0.629630
2    0.472826
3    0.242363
Name: Survived, dtype: float64 

By Embarked:
 Embarked
C    0.553571
Q    0.389610
S    0.336957
Name: Survived, dtype: float64


## 4. Title helper

Extract the title from `Name` and fold rare/synonymous titles into 5 clean groups: `Mr, Mrs, Miss, Master, Rare`.

In [5]:
# Map synonyms (Mlle->Miss, Mme->Mrs) and rare aristocratic/military titles -> Rare.
TITLE_MAP = {
    'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs',
    'Lady': 'Rare', 'Countess': 'Rare', 'Capt': 'Rare', 'Col': 'Rare',
    'Don': 'Rare', 'Dr': 'Rare', 'Major': 'Rare', 'Rev': 'Rare',
    'Sir': 'Rare', 'Jonkheer': 'Rare', 'Dona': 'Rare',
}

def extract_title(name):
    # 'Braund, Mr. Owen Harris' -> 'Mr'
    title = name.split(',')[1].split('.')[0].strip()
    return TITLE_MAP.get(title, title)

## 5. Family / Group survival feature (the key upgrade)

People travelling together tended to live or die together (the "women & children first" lifeboat dynamic acted at the *group* level). For each passenger we check their relatives (same **surname + fare**) and ticket-mates (same **ticket**) and read off whether *those other people* survived.

Rules that keep it honest (no leakage):
- We always **drop the passenger themselves** before looking — never use a person's own label.
- Test passengers have `Survived = NaN`, so a test passenger's value comes **only from train relatives**.
- Unknown groups default to `0.5` (neutral).

In [6]:
def family_survival(df, surv):
    """surv: Survived aligned to df.index, NaN where unknown (test / held-out fold)."""
    d = df.copy(); d['S'] = surv.values
    fs = pd.Series(0.5, index=d.index)                 # neutral default
    # 1) group by surname + fare (true families)
    for _, g in d.groupby(['Last', 'Fare']):
        if len(g) > 1:
            for i in g.index:
                others = g.drop(i)['S']                 # exclude self -> no self-leak
                if others.notna().any():
                    fs[i] = 1.0 if others.max() == 1 else 0.0
    # 2) fall back to ticket-mates for anyone still unknown
    for _, g in d.groupby('Ticket'):
        if len(g) > 1:
            for i in g.index:
                if fs[i] == 0.5:
                    others = g.drop(i)['S']
                    if others.notna().any():
                        fs[i] = 1.0 if others.max() == 1 else 0.0
    return fs.values

## 6. Feature engineering + missing values + encoding

One shared function so train and test are processed identically. Train+test are concatenated only to compute **fills** and **ticket-group sizes** (uses no target -> no leakage). The lean feature set was chosen because simpler models generalised better (smaller CV→leaderboard gap).

In [7]:
def build_features(train, test):
    """Return (X_train, X_test, y, df) — numeric, column-aligned matrices."""
    y = train['Survived'].astype(float).values

    # combine (test target stays NaN) for consistent fills / group stats
    tr = train.drop(columns=['Survived']).copy(); tr['__src__'] = 'train'
    te = test.copy();                              te['__src__'] = 'test'
    df = pd.concat([tr, te], ignore_index=True)

    df['Title'] = df['Name'].apply(extract_title)                 # 5 title groups
    df['Last']  = df['Name'].apply(lambda x: x.split(',')[0].strip())  # surname for groups

    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1             # incl. self
    df['IsChild_raw'] = df['Age']                               # placeholder, set after Age fill

    df['HasCabin'] = df['Cabin'].notna().astype(int)            # cabin recorded = wealthier/known

    # ticket group size across train+test (no target used)
    df['TGS'] = df.groupby('Ticket')['Ticket'].transform('count')

    # Fare: median by Pclass, then log to tame the skew
    df['Fare'] = df.groupby('Pclass')['Fare'].transform(lambda s: s.fillna(s.median()))
    df['FareLog'] = np.log1p(df['Fare'])

    # Age: median by Title+Pclass, with fallbacks (never one global mean)
    df['Age'] = df.groupby(['Title', 'Pclass'])['Age'].transform(lambda s: s.fillna(s.median()))
    df['Age'] = df.groupby(['Title'])['Age'].transform(lambda s: s.fillna(s.median()))
    df['Age'] = df['Age'].fillna(df['Age'].median())
    df['IsChild'] = (df['Age'] < 16).astype(int)               # children prioritised

    df['Sex'] = (df['Sex'] == 'male').astype(int)              # 1=male, 0=female
    return df, y


NUM = ['Sex', 'Age', 'FareLog', 'FamilySize', 'IsChild', 'HasCabin', 'FamSurv']
CAT = ['Title', 'Pclass']

def encode(df, rows_mask):
    # one-hot encode categoricals; numeric kept as-is
    enc = pd.get_dummies(df[NUM + CAT], columns=CAT, drop_first=False, dtype=float)
    return enc[rows_mask].reset_index(drop=True)


df, y = build_features(train, test)
ntr = len(train)
print('Combined frame:', df.shape, '| numeric+cat features:', NUM, '+', CAT)
df[['Name', 'Title', 'Last', 'FamilySize', 'Age', 'FareLog', 'HasCabin']].head()

Combined frame: (1309, 20) | numeric+cat features: ['Sex', 'Age', 'FareLog', 'FamilySize', 'IsChild', 'HasCabin', 'FamSurv'] + ['Title', 'Pclass']


,Name,Title,Last,FamilySize,Age,FareLog,HasCabin
0,"Braund, Mr. Owen Harris",Mr,Braund,2,22.0,2.110213,0
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",Mrs,Cumings,2,38.0,4.280593,1
2,"Heikkinen, Miss. Laina",Miss,Heikkinen,1,26.0,2.188856,0
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",Mrs,Futrelle,2,35.0,3.990834,1
4,"Allen, Mr. William Henry",Mr,Allen,1,35.0,2.202765,0


## 7. Threshold tuning via **fold-safe** repeated stratified CV

LinearRegression outputs a continuous score, so we need a cut-off. We average accuracy over **5-fold × 6-repeat = 30 folds** for each threshold. Crucially, `FamSurv` is **recomputed inside every fold** using only that fold's training labels — otherwise validation labels would leak through relatives and inflate the score.

In [8]:
def tune_threshold(df, y, ntr):
    thresholds = np.arange(0.20, 0.801, 0.01)
    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=6, random_state=RANDOM_STATE)
    tr_df = df.iloc[:ntr].reset_index(drop=True)
    acc_sum = np.zeros(len(thresholds)); n = 0
    for tr_idx, val_idx in cv.split(np.zeros(ntr), y):
        d = tr_df.copy()
        # FamSurv from TRAIN-FOLD labels only (val rows look NaN -> no leakage)
        surv = pd.Series(np.nan, index=d.index); surv.iloc[tr_idx] = y[tr_idx]
        d['FamSurv'] = family_survival(d, surv)
        X = encode(d, np.ones(ntr, bool)).values.astype(float)
        sc = StandardScaler().fit(X[tr_idx])
        m = LinearRegression().fit(sc.transform(X[tr_idx]), y[tr_idx])
        raw = m.predict(sc.transform(X[val_idx]))
        for i, t in enumerate(thresholds):
            acc_sum[i] += accuracy_score(y[val_idx], (raw >= t).astype(int))
        n += 1
    mean_acc = acc_sum / n
    best = int(np.argmax(mean_acc))
    return thresholds[best], mean_acc[best], thresholds, mean_acc


best_t, best_acc, thr, macc = tune_threshold(df, y, ntr)
print(f'Best CV threshold = {best_t:.2f}  |  honest CV accuracy = {best_acc:.4f}')
print('CV accuracy @0.50  =', round(macc[np.argmin(np.abs(thr - 0.50))], 4))

Best CV threshold = 0.53  |  honest CV accuracy = 0.8492
CV accuracy @0.50  = 0.847


## 8. Retrain on the full training set

For the final model, `FamSurv` uses **all train labels** for the train rows (self always excluded) and feeds **train labels into the test rows** (test labels are unknown / NaN).

In [9]:
# Full-data FamSurv: known = all train labels, test = NaN
surv_full = pd.Series(np.nan, index=df.index); surv_full.iloc[:ntr] = y
df['FamSurv'] = family_survival(df, surv_full)
print('Test FamSurv counts:', dict(pd.Series(np.round(df['FamSurv'].iloc[ntr:], 2)).value_counts()))

train_mask = df['__src__'].values == 'train'
test_mask  = df['__src__'].values == 'test'
X      = encode(df, train_mask)
X_test = encode(df, test_mask)
X, X_test = X.align(X_test, join='left', axis=1, fill_value=0)   # identical columns

Xv, Xtv = X.values.astype(float), X_test.values.astype(float)

# quick hold-out sanity check (optimistic re: FamSurv, just a smoke test)
X_tr, X_val, y_tr, y_val = train_test_split(Xv, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
sc = StandardScaler().fit(X_tr)
print('Hold-out acc @best_t:', round(accuracy_score(y_val,
      (LinearRegression().fit(sc.transform(X_tr), y_tr).predict(sc.transform(X_val)) >= best_t).astype(int)), 4))

# final fit on ALL training data
scaler = StandardScaler().fit(Xv)
model = LinearRegression().fit(scaler.transform(Xv), y)
test_raw = model.predict(scaler.transform(Xtv))
test_pred = (test_raw >= best_t).astype(int)
print('Predicted survivors:', int(test_pred.sum()), '/', len(test_pred))

Test FamSurv counts: {0.5: np.int64(247), 1.0: np.int64(96), 0.0: np.int64(75)}
Hold-out acc @best_t: 0.8156
Predicted survivors: 150 / 418


## 9. Create the Kaggle submission

In [10]:
# Exactly two columns, 418 rows + header.
submission = pd.DataFrame({'PassengerId': passenger_id, 'Survived': test_pred})
out_path = os.path.join(SUB, 'submission_linear_regression.csv')
submission.to_csv(out_path, index=False)

print('Saved:', out_path, '| shape:', submission.shape)
print(submission['Survived'].value_counts())
submission.head()

Saved: ../submissions/submission_linear_regression.csv | shape: (418, 2)
Survived
0    268
1    150
Name: count, dtype: int64


,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1


## 10. Conclusion (demo)

- Used **only `LinearRegression`** — no classifier anywhere.
- Baseline feature set scored **0.770** on Kaggle (CV ~0.83 → overfitting).
- Added an honest **Family/Group survival** feature (relatives'/ticket-mates' known outcomes, self always excluded, test driven only by train labels) and **trimmed to a lean feature set**.
- Tuned the threshold with **fold-safe** repeated CV → honest CV ≈ **0.85**.
- 171 / 418 test passengers receive a confident group signal, which is what lifts the leaderboard score.
- Output: `submissions/submission_linear_regression.csv` (418 rows).